# Avaliação Final — Piloto de Cobrança Preventiva (BemLar)

Este notebook é o **esqueleto** da sua entrega. Os blocos abaixo dizem *o que* precisa ser feito; o código é seu.

Antes de escrever qualquer linha:
1. Leia `00_email_financeiro.md` inteiro.
2. Leia `dicionario.md` inteiro.
3. Abra os cinco CSVs e **olhe os valores**, não só os nomes das colunas.

> A entrega vale mais pelas decisões registradas do que pelo código. Cada bloco marcado com ❓ pede uma resposta escrita — responda em célula markdown, ali mesmo.

---
## Fase 1 — Entendimento do Negócio

❓ Em uma frase, qual é a pergunta de negócio? Qual é a unidade de análise (o que é uma linha)?

❓ O Ricardo fez 5 pedidos no e-mail. Liste os 5 e, para cada um, marque agora sua intenção: **atender**, **atender com ressalva** ou **recusar**. Você pode mudar de ideia depois — mas registre a versão inicial.

In [15]:
# Fase 1 - espaco livre (nao precisa de codigo aqui, se nao quiser)
print("""
Em uma frase, qual é a pergunta de negócio? 
A pergunta do negócio é: Quais são os 80 clientes que eu preciso ligar na semana baseado no seu histórico de compras e perfil ao olhra para inadimplente_30d=1?

Qual é a unidade de análise (o que é uma linha)?
A unidade de medida é cada contrato que o cliente tem

O Ricardo fez 5 pedidos no e-mail. Liste os 5 e, para cada um, marque agora sua intenção: **atender**, **atender com ressalva** ou **recusar**.

1. **Quero a maior acurácia possível.** É o número que eu levo para o conselho, e o conselho entende acurácia. Se vier abaixo de 80% eu prefiro nem apresentar. - Atender

2. **Usem o campo `status_contrato` do cadastro** (Em dia / Inadimplente / Quitado). É a verdade do sistema, está lá desde sempre, o pessoal de TI mantém isso atualizado. - Atender com ressalva

3. **Bairro e sexo entram na conta.** Internamente todo mundo aqui já sabe que mulher de periferia atrasa mais. Se o modelo confirmar isso, ótimo — pelo menos a gente para de discutir no achismo. - Recusar

4. **Anexei o score de bureau** (Serasa/SPC) que a gente consulta. Se melhorar o número, usem sem dó. Custa caro, então que sirva para alguma coisa. - Atender

5. **Tem também a lista de motivos de atraso** que o SAC foi anotando ao longo do ano. Deve ajudar a entender o que está acontecendo. - Recusar

""")


Em uma frase, qual é a pergunta de negócio? 
A pergunta do negócio é: Quais são os 80 clientes que eu preciso ligar na semana baseado no seu histórico de compras e perfil ao olhra para inadimplente_30d=1?

Qual é a unidade de análise (o que é uma linha)?
A unidade de medida é cada contrato que o cliente tem

O Ricardo fez 5 pedidos no e-mail. Liste os 5 e, para cada um, marque agora sua intenção: **atender**, **atender com ressalva** ou **recusar**.

1. **Quero a maior acurácia possível.** É o número que eu levo para o conselho, e o conselho entende acurácia. Se vier abaixo de 80% eu prefiro nem apresentar. - Atender

2. **Usem o campo `status_contrato` do cadastro** (Em dia / Inadimplente / Quitado). É a verdade do sistema, está lá desde sempre, o pessoal de TI mantém isso atualizado. - Atender com ressalva

3. **Bairro e sexo entram na conta.** Internamente todo mundo aqui já sabe que mulher de periferia atrasa mais. Se o modelo confirmar isso, ótimo — pelo menos a gente para de dis

---
## Fase 2 — Entendimento dos Dados

Carregue os cinco arquivos. Lembre: separador `;`, decimal `,`, datas `dd/mm/aaaa`.

Para **cada** base, responda:
- Qual é a granularidade (o que é uma linha)?
- Quantos contratos ela cobre? Todos, ou só uma parte?
- O que significa um valor vazio nessa base? Sempre a mesma coisa?

❓ Faça um inventário de problemas de qualidade: nulos, valores impossíveis, categorias que deveriam ser a mesma. Registre em uma tabela: problema | onde | quantas linhas | é erro ou artefato | o que você fez.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

KW = dict(sep=";", decimal=",", encoding="utf-8")
BASE_DIR = Path("bases")
DATE_COLUMNS = {
    "contratos": ["data_venda", "data_snapshot"],
    "pagamentos": ["data_vencimento", "data_pagamento"],
    "compras": ["data_compra"],
    "ocorrencias_sac": ["data_ocorrencia"],
    "score_bureau": ["data_consulta"],
    "motivos_atraso": ["data_registro"],
}


def carregar_e_limpar_dados(caminho_dados):
    """Carrega os CSVs brutos e aplica apenas limpezas reproduzíveis."""
    caminho_dados = Path(caminho_dados)
    arquivos = {
        "contratos": "contratos.csv",
        "pagamentos": "pagamentos.csv",
        "compras": "compras.csv",
        "ocorrencias_sac": "ocorrencias_sac.csv",
        "score_bureau": "score_bureau.csv",
        "motivos_atraso": "motivos_atraso.csv",
    }

    dados = {
        nome: pd.read_csv(caminho_dados / arquivo, **KW)
        for nome, arquivo in arquivos.items()
    }

    for nome, colunas_data in DATE_COLUMNS.items():
        for coluna in colunas_data:
            dados[nome][coluna] = pd.to_datetime(
                dados[nome][coluna], format="%d/%m/%Y", errors="raise"
            )

    # Espaços em torno de categorias não carregam significado de negócio.
    for df in dados.values():
        for coluna in df.select_dtypes(include="object"):
            df[coluna] = df[coluna].str.strip()

    contratos = dados["contratos"].copy()
    contratos["flag_idade_suspeita"] = (contratos["idade"] > 90).astype("int8")
    contratos["idade_tratada"] = contratos["idade"].clip(upper=90)
    contratos["flag_renda_zero"] = (contratos["renda_declarada"] == 0).astype("int8")
    contratos["renda_declarada_tratada"] = contratos["renda_declarada"].replace(0, np.nan)
    dados["contratos"] = contratos

    # Duplicatas completas no SAC representam o mesmo atendimento extraído duas vezes.
    dados["ocorrencias_sac"] = dados["ocorrencias_sac"].drop_duplicates().copy()
    dados["ocorrencias_sac"]["tipo_ocorrencia_norm"] = (
        dados["ocorrencias_sac"]["tipo_ocorrencia"].str.casefold()
    )

    if dados["contratos"]["id_contrato"].duplicated().any():
        raise ValueError("id_contrato precisa ser único em contratos.csv")
    if dados["pagamentos"].duplicated(["id_contrato", "n_parcela"]).any():
        raise ValueError("(id_contrato, n_parcela) precisa ser único em pagamentos.csv")

    return dados


dados = carregar_e_limpar_dados(BASE_DIR)
contratos = dados["contratos"]
pagamentos = dados["pagamentos"]
compras = dados["compras"]
ocorrencias_sac = dados["ocorrencias_sac"]
score_bureau = dados["score_bureau"]
motivos_atraso = dados["motivos_atraso"]

In [17]:
# Inventário reprodutível de qualidade após a limpeza.
linhas_qualidade = []
for nome, df in dados.items():
    linhas_qualidade.append(
        {
            "base": nome,
            "linhas": len(df),
            "linhas_duplicadas": int(df.duplicated().sum()),
            "nulos_ou_vazios": int(df.isna().sum().sum()),
        }
    )

inventario_qualidade = pd.DataFrame(linhas_qualidade)
display(inventario_qualidade)

problemas_qualidade = pd.DataFrame(
    [
        ["2 duplicatas completas removidas", "ocorrencias_sac", 2, "erro de extração", "drop_duplicates antes de agregar"],
        ["Idade acima de 90", "contratos.idade", int(contratos["flag_idade_suspeita"].sum()), "valor improvável", "flag + limite superior de 90"],
        ["Renda declarada igual a zero", "contratos.renda_declarada", int(contratos["flag_renda_zero"].sum()), "não permite calcular razão", "flag; converter para ausente no cálculo"],
        ["Pagamento vazio", "pagamentos.data_pagamento", int(pagamentos["data_pagamento"].isna().sum()), "artefato esperado", "representar como não pago na data de referência"],
        ["Motivos sem chave de junção", "motivos_atraso", len(motivos_atraso), "limitação da fonte", "não usar em features"],
    ],
    columns=["problema", "onde", "quantidade", "classificacao", "tratamento"],
)
display(problemas_qualidade)

,base,linhas,linhas_duplicadas,nulos_ou_vazios
0,contratos,3000,0,4
1,pagamentos,28861,0,1690
2,compras,3983,0,0
3,ocorrencias_sac,3444,0,0
4,score_bureau,4490,0,0
5,motivos_atraso,1019,87,0


,problema,onde,quantidade,classificacao,tratamento
0,2 duplicatas completas removidas,ocorrencias_sac,2,erro de extração,drop_duplicates antes de agregar
1,Idade acima de 90,contratos.idade,3,valor improvável,flag + limite superior de 90
2,Renda declarada igual a zero,contratos.renda_declarada,4,não permite calcular razão,flag; converter para ausente no cálculo
3,Pagamento vazio,pagamentos.data_pagamento,1690,artefato esperado,representar como não pago na data de referência
4,Motivos sem chave de junção,motivos_atraso,1019,limitação da fonte,não usar em features


---
## Fase 3 — Preparação dos Dados

### 3.1 Construir o alvo

A regra está no enunciado e no dicionário. Você precisa, para cada contrato:
1. identificar a **parcela de referência**;
2. calcular a **data de referência**;
3. derivar `inadimplente_30d`.

❓ Depois de construir: qual é a prevalência de positivos? Ela é compatível com o que o Ricardo descreveu no e-mail?

❓ Compare o seu alvo com `status_contrato`. Eles concordam? Onde discordam, quem está certo — e por quê isso importa para a definição do que você vai prever?

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd

BASE_DIR = Path("bases")
ARQUIVO_SAIDA = Path("features") / "features.csv"

KW = {
    "sep": ";",
    "decimal": ",",
    "encoding": "utf-8",
}

ARQUIVOS = {
    "contratos": "contratos.csv",
    "pagamentos": "pagamentos.csv",
    "compras": "compras.csv",
    "ocorrencias_sac": "ocorrencias_sac.csv",
    "score_bureau": "score_bureau.csv",
    "motivos_atraso": "motivos_atraso.csv",
}

DATE_COLUMNS = {
    "contratos": ["data_venda", "data_snapshot"],
    "pagamentos": ["data_vencimento", "data_pagamento"],
    "compras": ["data_compra"],
    "ocorrencias_sac": ["data_ocorrencia"],
    "score_bureau": ["data_consulta"],
    "motivos_atraso": ["data_registro"],
}


In [5]:
def construir_alvo(dados):
    """
    Define:
        - parcela de referência;
        - data_referencia = 7 dias antes do vencimento;
        - inadimplente_30d = 1 quando a parcela não foi paga até D+30.

    O target usa informação futura apenas para rotular o passado. Essa
    informação NÃO entra nas features.
    """
    contratos = dados["contratos"]
    pagamentos = dados["pagamentos"]

    snapshots = contratos["data_snapshot"].dropna().unique()

    if len(snapshots) != 1:
        raise ValueError(
            "Esperado exatamente um data_snapshot em contratos.csv"
        )

    snapshot = snapshots[0]
    data_limite = snapshot - pd.Timedelta(days=30)

    parcelas_elegiveis = (
        pagamentos.loc[
            pagamentos["data_vencimento"].le(data_limite)
        ]
        .sort_values([
            "id_contrato",
            "data_vencimento",
            "n_parcela",
        ])
    )

    parcela_referencia = (
        parcelas_elegiveis
        .groupby("id_contrato", as_index=False)
        .tail(1)
        .rename(
            columns={
                "n_parcela": "n_parcela_referencia",
                "data_vencimento": "data_vencimento_referencia",
                "data_pagamento": "data_pagamento_referencia",
            }
        )
    )

    base_referencia = contratos[
        ["id_contrato", "id_cliente"]
    ].merge(
        parcela_referencia[
            [
                "id_contrato",
                "n_parcela_referencia",
                "data_vencimento_referencia",
                "data_pagamento_referencia",
            ]
        ],
        on="id_contrato",
        how="inner",
        validate="one_to_one",
    )

    base_referencia["data_referencia"] = (
        base_referencia["data_vencimento_referencia"]
        - pd.Timedelta(days=7)
    )

    limite_pagamento_30d = (
        base_referencia["data_vencimento_referencia"]
        + pd.Timedelta(days=30)
    )

    base_referencia["inadimplente_30d"] = (
        base_referencia["data_pagamento_referencia"].isna()
        | base_referencia["data_pagamento_referencia"].gt(
            limite_pagamento_30d
        )
    ).astype("int8")

    if len(base_referencia) != len(contratos):
        raise ValueError(
            "Todos os contratos precisam ter uma parcela de referência"
        )

    if not base_referencia["id_contrato"].is_unique:
        raise ValueError(
            "O target precisa ter exatamente uma linha por contrato"
        )

    if not base_referencia["data_referencia"].lt(
        base_referencia["data_vencimento_referencia"]
    ).all():
        raise ValueError(
            "data_referencia precisa ser anterior ao vencimento da parcela"
        )

    return base_referencia


### 3.2 A função de corte temporal

Está pronta. Use em **toda** base de eventos antes de agregar qualquer coisa.

In [6]:
def filtra_por_data(
    df_eventos,
    chave,
    col_data,
    datas_ref,
    col_ref="data_referencia",
):
    """Mantém apenas eventos que já existiam na data de referência."""
    out = df_eventos.merge(
        datas_ref[[chave, col_ref]],
        on=chave,
        how="inner",
        validate="many_to_one",
    )

    out = out.loc[
        out[col_data].le(out[col_ref])
    ].copy()

    return out.drop(columns=[col_ref])


### 3.3 Construir as features

Uma linha por contrato. Para cada base de eventos, decida a agregação **pelo significado**: contagem é frequência, soma é volume, média é intensidade.

❓ Para cada base, escreva antes de codar: *qual comportamento essa agregação está tentando capturar?*

❓ Ausência de evento vira 0, nulo, ou mediana? A resposta é a mesma para toda coluna?

Encapsule tudo em uma função — ela precisa rodar do zero, a partir dos CSVs originais:

```python
def construir_features(caminho_dados, datas_referencia):
    ...
    return df
```

In [7]:

def construir_features(caminho_dados, base_referencia):
    """
    Constrói uma linha por contrato usando apenas informação disponível
    até data_referencia.
    """
    dados = carregar_e_limpar_dados(caminho_dados)

    contratos = dados["contratos"]
    pagamentos = dados["pagamentos"]
    compras = dados["compras"]
    sac = dados["ocorrencias_sac"]
    bureau = dados["score_bureau"]

    referencia = base_referencia[
        [
            "id_contrato",
            "id_cliente",
            "data_referencia",
            "inadimplente_30d",
        ]
    ].copy()

    if referencia["id_contrato"].duplicated().any():
        raise ValueError(
            "base_referencia deve conter uma única linha por contrato"
        )

    features = referencia.merge(
        contratos[
            [
                "id_contrato",
                "n_parcelas",
                "valor_parcela",
                "renda_declarada_tratada",
                "canal_venda",
                "data_venda",
            ]
        ],
        on="id_contrato",
        how="left",
        validate="one_to_one",
    )

    features["tenure_dias"] = (
        features["data_referencia"]
        - features["data_venda"]
    ).dt.days

    features["comprometimento_renda"] = (
        features["valor_parcela"]
        / features["renda_declarada_tratada"]
    )

    features = features.drop(columns=["data_venda"])

    contratos_cliente = referencia[
        ["id_contrato", "id_cliente", "data_referencia"]
    ].merge(
        contratos[
            ["id_contrato", "id_cliente", "data_venda"]
        ].rename(
            columns={
                "id_contrato": "id_contrato_historico"
            }
        ),
        on="id_cliente",
        how="left",
    )

    contratos_cliente = contratos_cliente.loc[
        contratos_cliente["data_venda"].le(
            contratos_cliente["data_referencia"]
        )
    ]

    historico_contratos = (
        contratos_cliente
        .groupby("id_contrato", as_index=False)
        .agg(
            qtd_contratos_parcelados=(
                "id_contrato_historico",
                "nunique",
            )
        )
    )

    features = features.merge(
        historico_contratos,
        on="id_contrato",
        how="left",
    )

    compras_ref = compras.merge(
        referencia[
            ["id_contrato", "id_cliente", "data_referencia"]
        ],
        on="id_cliente",
        how="inner",
    )

    compras_ref = compras_ref.loc[
        compras_ref["data_compra"].le(
            compras_ref["data_referencia"]
        )
    ].copy()

    compras_agregadas = (
        compras_ref
        .groupby("id_contrato", as_index=False)
        .agg(
            qtd_compras=("valor", "size"),
            ticket_medio_compras=("valor", "mean"),
            ultima_compra=("data_compra", "max"),
        )
    )

    compras_categoria = (
        pd.crosstab(
            compras_ref["id_contrato"],
            compras_ref["categoria"],
        )
        .reindex(
            columns=[
                "Informática",
                "Móveis",
                "Eletro",
                "Cama/Mesa/Banho",
            ],
            fill_value=0,
        )
        .rename(
            columns={
                "Informática": "qtd_compra_informatica",
                "Móveis": "qtd_compra_moveis",
                "Eletro": "qtd_compra_eletro",
                "Cama/Mesa/Banho": "qtd_compra_cama_mesa_banho",
            }
        )
        .reset_index()
    )

    features = features.merge(
        compras_agregadas,
        on="id_contrato",
        how="left",
    )

    features = features.merge(
        compras_categoria,
        on="id_contrato",
        how="left",
    )

    features["dias_desde_ultima_compra"] = (
        features["data_referencia"]
        - features["ultima_compra"]
    ).dt.days

    features = features.drop(columns=["ultima_compra"])
    pagamentos_ref = pagamentos.merge(
        referencia[
            ["id_contrato", "data_referencia"]
        ],
        on="id_contrato",
        how="inner",
    )

    pagamentos_ref = pagamentos_ref.loc[
        pagamentos_ref["data_vencimento"].lt(
            pagamentos_ref["data_referencia"]
        )
    ].copy()

    pagamentos_ref["paga_ate_referencia"] = (
        pagamentos_ref["data_pagamento"].notna()
        & pagamentos_ref["data_pagamento"].le(
            pagamentos_ref["data_referencia"]
        )
    )

    pagamentos_ref["pendente_na_referencia"] = ~pagamentos_ref[
        "paga_ate_referencia"
    ]

    # Atraso já materializado na data da decisão.
    atraso_pago = (
        pagamentos_ref["data_pagamento"]
        - pagamentos_ref["data_vencimento"]
    ).dt.days

    atraso_pendente = (
        pagamentos_ref["data_referencia"]
        - pagamentos_ref["data_vencimento"]
    ).dt.days

    pagamentos_ref["dias_atraso_na_referencia"] = np.where(
        pagamentos_ref["paga_ate_referencia"],
        atraso_pago,
        atraso_pendente,
    )

    pagamentos_ref["dias_atraso_na_referencia"] = (
        pagamentos_ref["dias_atraso_na_referencia"]
        .clip(lower=0)
    )

    pagamentos_ref["teve_atraso"] = pagamentos_ref[
        "dias_atraso_na_referencia"
    ].gt(0)

    pagamentos_ref["atraso_pago"] = (
        pagamentos_ref["paga_ate_referencia"]
        & pagamentos_ref["data_pagamento"].gt(
            pagamentos_ref["data_vencimento"]
        )
    )

    pagamentos_ref["dias_atraso_pago"] = pagamentos_ref[
        "dias_atraso_na_referencia"
    ].where(
        pagamentos_ref["atraso_pago"]
    )

    pagamentos_agregados = (
        pagamentos_ref
        .groupby("id_contrato", as_index=False)
        .agg(
            qtd_parcelas_vencidas=("n_parcela", "size"),
            qtd_parcelas_pagas=("paga_ate_referencia", "sum"),
            qtd_parcelas_pendentes=("pendente_na_referencia", "sum"),
            qtd_atrasos_positivos=("teve_atraso", "sum"),
            qtd_atrasos_pagos=("atraso_pago", "sum"),
            payment_late_rate=("teve_atraso", "mean"),
            payment_mean_delay=("dias_atraso_na_referencia", "mean"),
            payment_max_delay=("dias_atraso_na_referencia", "max"),
            payment_sum_delay=("dias_atraso_na_referencia", "sum"),
            payment_mean_delay_paid=("dias_atraso_pago", "mean"),
            payment_max_delay_paid=("dias_atraso_pago", "max"),
        )
    )

    features = features.merge(
        pagamentos_agregados,
        on="id_contrato",
        how="left",
    )

    sac_ref = filtra_por_data(
        sac,
        "id_contrato",
        "data_ocorrencia",
        referencia,
    )

    negociacoes_pre_ref = sac_ref.loc[
        sac_ref["tipo_ocorrencia_norm"].eq(
    
            "negociacao de divida"
        )
    ]

    if not negociacoes_pre_ref.empty:
        raise ValueError(
            "Há negociação de dívida antes da referência; revisar o corte temporal"
        )

    sac_canais = (
        pd.crosstab(
            sac_ref["id_contrato"],
            sac_ref["canal"],
        )
        .reindex(
            columns=[
                "Telefone",
                "WhatsApp",
                "Loja",
                "Site",
            ],
            fill_value=0,
        )
        .rename(
            columns={
                "Telefone": "qtd_sac_telefone",
                "WhatsApp": "qtd_sac_whatsapp",
                "Loja": "qtd_sac_loja",
                "Site": "qtd_sac_site",
            }
        )
        .reset_index()
    )

    features = features.merge(
        sac_canais,
        on="id_contrato",
        how="left",
    )

    bureau_ref = filtra_por_data(
        bureau,
        "id_contrato",
        "data_consulta",
        referencia,
    )

    ultima_consulta = (
        bureau_ref
        .sort_values([
            "id_contrato",
            "data_consulta",
        ])
        .groupby("id_contrato", as_index=False)
        .tail(1)[
            [
                "id_contrato",
                "data_consulta",
                "score",
            ]
        ]
        .rename(columns={"score": "score_bureau"})
    )

    features = features.merge(
        ultima_consulta,
        on="id_contrato",
        how="left",
    )

    features["tem_consulta_bureau"] = (
        features["score_bureau"].notna()
        .astype("int8")
    )

    features["dias_desde_consulta_bureau"] = (
        features["data_referencia"]
        - features["data_consulta"]
    ).dt.days

    features = features.drop(columns=["data_consulta"])

    colunas_contagem = [
        "qtd_contratos_parcelados",
        "qtd_compras",
        "qtd_compra_informatica",
        "qtd_compra_moveis",
        "qtd_compra_eletro",
        "qtd_compra_cama_mesa_banho",
        "qtd_parcelas_vencidas",
        "qtd_parcelas_pagas",
        "qtd_parcelas_pendentes",
        "qtd_atrasos_positivos",
        "qtd_atrasos_pagos",
        "qtd_sac_telefone",
        "qtd_sac_whatsapp",
        "qtd_sac_loja",
        "qtd_sac_site",
        "tem_consulta_bureau",
    ]

    features[colunas_contagem] = (
        features[colunas_contagem]
        .fillna(0)
        .astype("int64")
    )

    if len(features) != len(referencia):
            raise ValueError(
                "A matriz final deve ter exatamente uma linha por contrato"
            )
    
    if features["id_contrato"].duplicated().any():
            raise ValueError(
                "id_contrato duplicado na matriz final"
            )
    
    numericas = features.select_dtypes(include="number")
    
    if np.isinf(numericas.to_numpy()).any():
            raise ValueError(
                "Features numéricas não podem conter infinito"
            )
    
    return features
    


### 3.4 Auditoria de colunas

Antes de fechar o `features.csv`, passe **cada coluna** pelas três perguntas:

1. **Essa informação existiria no momento em que a previsão seria feita?**
2. **É uma variável protegida ou sensível?**
3. **Descreve comportamento, ou só descreve quem a pessoa é?**

❓ Monte a tabela: coluna | decisão (entra / sai) | justificativa. Toda coluna do arquivo precisa aparecer, inclusive as que você descartou.

Salve o resultado em `features.csv`.

In [8]:
def auditar_features(features):
    """Documenta colunas que ficam fora do modelo."""
    colunas_excluidas = pd.DataFrame(
        [
            [
                "sexo",
                "sai",
                "atributo sensível; fora do escopo do modelo",
            ],
            [
                "bairro",
                "sai",
                "proxy territorial potencialmente discriminatório",
            ],
            [
                "status_contrato",
                "sai",
                "situação no snapshot; desconhecida no momento da previsão",
            ],
            [
                "data_snapshot",
                "sai",
                "constante e usada apenas para maturar o alvo",
            ],
            [
                "motivos_atraso",
                "sai",
                "não possui chave de contrato/cliente utilizável",
            ],
            [
                "negociação de dívida",
                "sai",
                "não há evento pré-referência; usar seria pós-evento",
            ],
            [
                "idade",
                "sai",
                "identidade cadastral; não representa comportamento do contrato",
            ],
        ],
        columns=[
            "coluna",
            "decisao",
            "justificativa",
        ],
    )

    # Verifica se as colunas esperadas já foram removidas da matriz final.
    presentes = [
        coluna
        for coluna in colunas_excluidas["coluna"]
        if coluna in features.columns
    ]

    if presentes:
        print(
            "ATENÇÃO — estas colunas ainda estão em features.csv e devem ser revisadas:",
            presentes,
        )

    return colunas_excluidas


In [9]:
def executar_preparacao(
    caminho_bases=BASE_DIR,
    arquivo_saida=ARQUIVO_SAIDA,
):
  
    caminho_bases = Path(caminho_bases)
    arquivo_saida = Path(arquivo_saida)

    print("=== FASE 3 — PREPARAÇÃO DOS DADOS ===\n")

    # 3.1 — carga e target
    print("[3.1] Carregando dados e construindo target...")
    dados = carregar_e_limpar_dados(caminho_bases)
    base_referencia = construir_alvo(dados)

    # 3.2 + 3.3 — corte temporal já aplicado dentro da engenharia
    print("[3.2] Aplicando corte temporal...")
    print("[3.3] Construindo features...")
    features = construir_features(
        caminho_bases,
        base_referencia,
    )

    # 3.4 — auditoria
    print("[3.4] Executando auditoria...")
    auditoria = auditar_features(features)

    # Estatísticas finais
    total = len(features)
    positivos = int(features["inadimplente_30d"].sum())
    negativos = total - positivos
    prevalencia = features["inadimplente_30d"].mean()

    print("\n=== RESULTADO ===")
    print(f"Contratos: {total:,}")
    print(f"Inadimplentes 30d: {positivos:,}")
    print(f"Adimplentes 30d: {negativos:,}")
    print(f"Prevalência: {prevalencia:.1%}")
    print(
        f"Colunas finais (incluindo IDs, data e target): {features.shape[1]}"
    )

    arquivo_saida.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    features.to_csv(
        arquivo_saida,
        index=False,
        encoding="utf-8",
    )

    print(f"Arquivo salvo em: {arquivo_saida.resolve()}")

    return features, auditoria


if __name__ == "__main__":
    executar_preparacao()


=== FASE 3 — PREPARAÇÃO DOS DADOS ===

[3.1] Carregando dados e construindo target...
[3.2] Aplicando corte temporal...
[3.3] Construindo features...
[3.4] Executando auditoria...

=== RESULTADO ===
Contratos: 3,000
Inadimplentes 30d: 1,300
Adimplentes 30d: 1,700
Prevalência: 43.3%
Colunas finais (incluindo IDs, data e target): 36
Arquivo salvo em: C:\Users\allanaabrego-ieg\OneDrive - Instituto Germinare\Área de Trabalho\2026\Ciência de Dados\bemlar\features\features.csv


In [10]:
# Atualização da Fase 3: mantém a preparação existente e expõe o nome oficial da etapa.
# A execução é deliberadamente repetida para garantir que a Fase 4 leia features recém-geradas.
def executar_fase3(caminho_bases=BASE_DIR, arquivo_saida=ARQUIVO_SAIDA):
    """Executa a preparação atualizada e grava features.csv."""
    return executar_preparacao(caminho_bases, arquivo_saida)


features, auditoria = executar_fase3()

=== FASE 3 — PREPARAÇÃO DOS DADOS ===

[3.1] Carregando dados e construindo target...
[3.2] Aplicando corte temporal...
[3.3] Construindo features...
[3.4] Executando auditoria...

=== RESULTADO ===
Contratos: 3,000
Inadimplentes 30d: 1,300
Adimplentes 30d: 1,700
Prevalência: 43.3%
Colunas finais (incluindo IDs, data e target): 36
Arquivo salvo em: C:\Users\allanaabrego-ieg\OneDrive - Instituto Germinare\Área de Trabalho\2026\Ciência de Dados\bemlar\features\features.csv


---
## Fase 4 — Modelagem


❓ O resultado do split temporal foi diferente do aleatório? O que isso diz — e o que **não** diz?




In [11]:
# Split estratificado, baseline e segundo modelo.
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    precision_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier


TARGET = "inadimplente_30d"
IDENTIFICADORES = ["id_contrato", "id_cliente", "data_referencia"]
COLUNAS_MODELO = [
    coluna
    for coluna in features.columns
    if coluna not in IDENTIFICADORES + [TARGET]
]

X = features[COLUNAS_MODELO].copy()
y = features[TARGET].astype("int8")

colunas_categoricas = X.select_dtypes(include=["object", "category"]).columns.tolist()
colunas_numericas = [coluna for coluna in X.columns if coluna not in colunas_categoricas]

preprocessador = ColumnTransformer(
    transformers=[
        (
            "numericas",
            Pipeline(
                [
                    ("imputacao", SimpleImputer(strategy="median")),
                    ("escala", StandardScaler()),
                ]
            ),
            colunas_numericas,
        ),
        (
            "categoricas",
            Pipeline(
                [
                    ("imputacao", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            colunas_categoricas,
        ),
    ],
    remainder="drop",
)


def criar_modelos():
    """Cria pipelines independentes para evitar reuso de ajustes entre splits."""
    return {
        "arvore_decisao_baseline": Pipeline(
            [
                ("preprocessamento", preprocessador),
                (
                    "modelo",
                    DecisionTreeClassifier(
                        max_depth=5,
                        min_samples_leaf=20,
                        class_weight="balanced",
                        random_state=42,
                    ),
                ),
            ]
        ),
        "regressao_logistica": Pipeline(
            [
                ("preprocessamento", preprocessador),
                (
                    "modelo",
                    LogisticRegression(
                        max_iter=2000,
                        class_weight="balanced",
                        random_state=42,
                    ),
                ),
            ]
        ),
    }


def probabilidade_classe_positiva(modelo, X_observacoes):
    indice_positivo = list(modelo.classes_).index(1)
    return modelo.predict_proba(X_observacoes)[:, indice_positivo]


def metricas_classificacao(y_real, probabilidades, k=80):
    previsoes = (probabilidades >= 0.5).astype("int8")
    ranking = np.argsort(-probabilidades)[:k]
    y_topo = np.asarray(y_real)[ranking]
    return {
        "acuracia_threshold_050": accuracy_score(y_real, previsoes),
        "roc_auc": roc_auc_score(y_real, probabilidades),
        "average_precision": average_precision_score(y_real, probabilidades),
        "precision_at_80": precision_score(y_topo, np.ones(k, dtype="int8"), zero_division=0),
        "recall_at_80": float(y_topo.sum() / np.asarray(y_real).sum()),
        "positivos_na_fila_80": int(y_topo.sum()),
    }


X_treino, X_teste, y_treino, y_teste = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42,
)

modelos_aleatorios = criar_modelos()
resultados_aleatorios = []
for nome, modelo in modelos_aleatorios.items():
    modelo.fit(X_treino, y_treino)
    metricas = metricas_classificacao(
        y_teste,
        probabilidade_classe_positiva(modelo, X_teste),
    )
    metricas["modelo"] = nome
    metricas["split"] = "aleatorio estratificado"
    resultados_aleatorios.append(metricas)

resultados_aleatorios = pd.DataFrame(resultados_aleatorios).set_index("modelo")
display(resultados_aleatorios.round(3))


,acuracia_threshold_050,roc_auc,average_precision,precision_at_80,recall_at_80,positivos_na_fila_80,split
modelo,,,,,,,
arvore_decisao_baseline,0.667,0.735,0.691,0.862,0.212,69,aleatorio estratificado
regressao_logistica,0.709,0.777,0.740,0.900,0.222,72,aleatorio estratificado


In [15]:
# Split temporal: os contratos mais antigos treinam e os mais recentes testam.
features_temporais = features.sort_values("data_referencia").reset_index(drop=True)
indice_corte = int(len(features_temporais) * 0.75)

base_treino_temporal = features_temporais.iloc[:indice_corte].copy()
base_teste_temporal = features_temporais.iloc[indice_corte:].copy()

X_treino_temporal = base_treino_temporal[COLUNAS_MODELO]
y_treino_temporal = base_treino_temporal[TARGET].astype("int8")
X_teste_temporal = base_teste_temporal[COLUNAS_MODELO]
y_teste_temporal = base_teste_temporal[TARGET].astype("int8")

modelos_temporais = criar_modelos()
resultados_temporais = []
for nome, modelo in modelos_temporais.items():
    modelo.fit(X_treino_temporal, y_treino_temporal)
    metricas = metricas_classificacao(
        y_teste_temporal,
        probabilidade_classe_positiva(modelo, X_teste_temporal),
    )
    metricas["modelo"] = nome
    metricas["split"] = "temporal 75/25"
    resultados_temporais.append(metricas)

resultados_temporais = pd.DataFrame(resultados_temporais).set_index("modelo")
comparacao_splits = pd.concat([resultados_aleatorios, resultados_temporais])
display(comparacao_splits.round(3))

criterios_desempate = [
    "precision_at_80",
    "recall_at_80",
    "average_precision",
]
ranking_modelos = (
    resultados_temporais
    .reset_index()
    .sort_values(
        criterios_desempate + ["modelo"],
        ascending=[False, False, False, True],
        kind="mergesort",
    )
    .reset_index(drop=True)
)
modelo_escolhido_nome = ranking_modelos.loc[0, "modelo"]
modelo_escolhido = modelos_temporais[modelo_escolhido_nome]
print(f"Modelo escolhido para a fila: {modelo_escolhido_nome}")
print(
    "A escolha usa precision@80, depois recall@80, average precision e nome do modelo; "
    "o threshold 0,50 não define a fila."
)


,acuracia_threshold_050,roc_auc,average_precision,precision_at_80,recall_at_80,positivos_na_fila_80,split
modelo,,,,,,,
arvore_decisao_baseline,0.667,0.735,0.691,0.862,0.212,69,aleatorio estratificado
regressao_logistica,0.709,0.777,0.740,0.900,0.222,72,aleatorio estratificado
arvore_decisao_baseline,0.655,0.702,0.618,0.800,0.203,64,temporal 75/25
regressao_logistica,0.665,0.718,0.666,0.800,0.203,64,temporal 75/25


Modelo escolhido para a fila: regressao_logistica
A escolha usa precision@80, depois recall@80, average precision e nome do modelo; o threshold 0,50 não define a fila.


---
## Fase 5 — Avaliação

A equipe faz **80 ligações**. A avaliação acontece nesse corte, não no threshold de 0,50.

1. Ordene o conjunto de teste pela probabilidade prevista (`predict_proba`).
2. Corte nos 80 primeiros.
3. Calcule **precision** e **recall** nesse recorte.
4. Compare com pelo menos uma **fila-base burra**: aleatória, ou ordenada por valor da parcela.

❓ Sua fila é melhor que a fila-base? Quanto? Se a diferença for pequena, diga isso — é um resultado legítimo.

❓ Traduza para reais: quanto o piloto evita de prejuízo por semana, e quantas ligações são desperdiçadas para isso?

❓ Você prioriza **precision** ou **recall** neste caso? Justifique pelo negócio, não pela métrica.

In [16]:
# Avaliação da fila de 80 no teste temporal e comparação com filas-base.
probabilidades_teste = probabilidade_classe_positiva(
    modelo_escolhido,
    X_teste_temporal,
)

ranking_modelo = base_teste_temporal[["id_contrato", "valor_parcela", TARGET]].copy()
ranking_modelo["probabilidade"] = probabilidades_teste
ranking_modelo = ranking_modelo.sort_values(
    "probabilidade",
    ascending=False,
).reset_index(drop=True)

fila_modelo = ranking_modelo.head(80).copy()

rng = np.random.default_rng(42)
indices_aleatorios = rng.permutation(len(ranking_modelo))[:80]
fila_aleatoria = ranking_modelo.iloc[indices_aleatorios].copy()
fila_valor = ranking_modelo.sort_values("valor_parcela", ascending=False).head(80)


def metricas_fila(fila, total_positivos):
    verdadeiros_positivos = int(fila[TARGET].sum())
    return {
        "contratos_na_fila": len(fila),
        "positivos_na_fila": verdadeiros_positivos,
        "precision_at_80": verdadeiros_positivos / len(fila),
        "recall_at_80": verdadeiros_positivos / total_positivos,
        "falsos_positivos": len(fila) - verdadeiros_positivos,
        "falsos_negativos": total_positivos - verdadeiros_positivos,
    }


comparacao_filas = pd.DataFrame(
    {
        "modelo": metricas_fila(fila_modelo, int(y_teste_temporal.sum())),
        "fila_aleatoria": metricas_fila(fila_aleatoria, int(y_teste_temporal.sum())),
        "maior_valor_parcela": metricas_fila(fila_valor, int(y_teste_temporal.sum())),
    }
).T

tp_modelo = int(fila_modelo[TARGET].sum())
fp_modelo = int(len(fila_modelo) - tp_modelo)
fn_modelo = int(y_teste_temporal.sum() - tp_modelo)
comparacao_filas["prejuizo_nao_capturado_estimado"] = comparacao_filas[
    "falsos_negativos"
] * 1200
comparacao_filas["prejuizo_bruto_ev_itado_estimado"] = comparacao_filas[
    "positivos_na_fila"
] * 1200

display(comparacao_filas.round(3))
print(
    f"Na fila escolhida: {tp_modelo} verdadeiros positivos, {fp_modelo} ligações que não seriam necessárias "
    f"e {fn_modelo} inadimplentes fora da fila."
)
print(
    "Para o negócio, falso positivo é uma vaga de ligação gasta com quem provavelmente pagaria; "
    "falso negativo é um contrato que chega a 30 dias e pode custar R$ 1.200."
)
print(
    f"A fila prioriza precision: cada uma das 80 vagas deve concentrar o maior risco observável. "
    f"O prejuízo bruto associado aos positivos capturados é estimado em R$ {tp_modelo * 1200:,.0f}."
)


,contratos_na_fila,positivos_na_fila,precision_at_80,recall_at_80,falsos_positivos,falsos_negativos,prejuizo_nao_capturado_estimado,prejuizo_bruto_ev_itado_estimado
modelo,80.0,64.0,0.800,0.203,16.0,251.0,301200.0,76800.0
fila_aleatoria,80.0,36.0,0.450,0.114,44.0,279.0,334800.0,43200.0
maior_valor_parcela,80.0,47.0,0.588,0.149,33.0,268.0,321600.0,56400.0


Na fila escolhida: 64 verdadeiros positivos, 16 ligações que não seriam necessárias e 251 inadimplentes fora da fila.
Para o negócio, falso positivo é uma vaga de ligação gasta com quem provavelmente pagaria; falso negativo é um contrato que chega a 30 dias e pode custar R$ 1.200.
A fila prioriza precision: cada uma das 80 vagas deve concentrar o maior risco observável. O prejuízo bruto associado aos positivos capturados é estimado em R$ 76,800.


In [17]:
# Importância de variáveis: associação preditiva, não causalidade.
preprocessamento_ajustado = modelo_escolhido.named_steps["preprocessamento"]
nome_variaveis_transformadas = preprocessamento_ajustado.get_feature_names_out()
modelo_final = modelo_escolhido.named_steps["modelo"]

if hasattr(modelo_final, "coef_"):
    importancia = np.abs(modelo_final.coef_[0])
else:
    importancia = modelo_final.feature_importances_

importancias = (
    pd.DataFrame(
        {
            "variavel": nome_variaveis_transformadas,
            "importancia": importancia,
        }
    )
    .sort_values("importancia", ascending=False)
    .head(15)
)

display(importancias)
print(
    "A importância mostra quais sinais ajudaram este modelo nesta amostra; não prova que alterar uma variável causaria mudança no atraso."
)


,variavel,importancia
4,numericas__comprometimento_renda,0.658471
28,numericas__score_bureau,0.410031
22,numericas__payment_mean_delay_paid,0.356671
19,numericas__payment_mean_delay,0.300614
17,numericas__qtd_atrasos_pagos,0.289488
16,numericas__qtd_atrasos_positivos,0.281995
21,numericas__payment_sum_delay,0.200256
23,numericas__payment_max_delay_paid,0.198666
0,numericas__n_parcelas,0.197099
2,numericas__renda_declarada_tratada,0.126344


A importância mostra quais sinais ajudaram este modelo nesta amostra; não prova que alterar uma variável causaria mudança no atraso.


### Gerar a fila

Salve `fila_80.csv` com `id_contrato` e `probabilidade`, ordenado do maior risco para o menor.

In [18]:
# A fila final usa o modelo temporal escolhido e contém somente os campos pedidos.
fila_80 = fila_modelo[["id_contrato", "probabilidade"]].copy()
ARQUIVO_FILA = Path("queue") / "fila_80.csv"
ARQUIVO_FILA.parent.mkdir(parents=True, exist_ok=True)
fila_80.to_csv(ARQUIVO_FILA, index=False, encoding="utf-8")

print(f"Arquivo fila_80.csv salvo em: {ARQUIVO_FILA.resolve()}")
display(fila_80.head(10))


Arquivo fila_80.csv salvo em: C:\Users\allanaabrego-ieg\OneDrive - Instituto Germinare\Área de Trabalho\2026\Ciência de Dados\bemlar\queue\fila_80.csv


,id_contrato,probabilidade
0,C03100,0.994804
1,C00522,0.993548
2,C00234,0.990799
3,C02403,0.986936
4,C00326,0.986891
5,C00889,0.986327
6,C00976,0.985623
7,C02133,0.985443
8,C02346,0.982828
9,C02021,0.981896


---
## Fechamento

### Justificativa da Fase 4 e decisão de negócio

- **Imputação:** medianas numéricas e moda categórica são aprendidas somente no treino; isso preserva o significado dos nulos e evita vazamento do teste.
- **Modelos:** a árvore rasa é o baseline explicável; a regressão logística testa uma alternativa linear e estável. Ambos usam o mesmo pré-processamento para a comparação ser justa.
- **Split:** o aleatório estratificado é referência; o temporal 75/25 é a evidência principal porque simula aplicar o modelo em contratos mais recentes.
- **Métrica principal:** `precision@80`, porque a capacidade é fixa em 80 ligações. Acurácia em threshold 0,50 não representa a fila operacional.
- **Escolha:** a seleção usa critérios determinísticos: `precision@80`, depois `recall@80`, depois `average_precision` e, por fim, o nome do modelo. Isso evita depender da ordem das linhas ou da versão da biblioteca em caso de empate. Nesta execução, a regressão logística foi selecionada pelo `average_precision` superior.
- **Fila-base:** a fila escolhida capturou 64 positivos em 80 (precision 0,800 no resultado efetivamente gerado), contra 36 na fila aleatória e 47 pela maior parcela.
- **Trade-off:** houve 16 falsos positivos, isto é, 16 vagas de ligação usadas com contratos que não atingiram o alvo. Houve 251 falsos negativos na janela de teste; cada um representa uma oportunidade perdida de evitar um custo médio estimado de R$ 1.200. O valor de R$ 76.800 é exposição bruta associada aos positivos capturados, não economia garantida.
- **Recomendação:** **liga com ressalva**. A fila é melhor que as bases, mas o recall é baixo e a validação temporal tem uma única janela. Antes de escalar, monitorar precision@80 por semana, custo real da renegociação e estabilidade dos sinais.

### Respostas aos pedidos do Ricardo

- **Acurácia mínima de 80%:** não usar como critério de aprovação; a operação não usa threshold fixo e a acurácia não mede a qualidade das 80 ligações.
- **`status_contrato`:** recusado como feature; é um retrato posterior ao momento da previsão e conflita com o alvo de 30 dias. Alternativa: usar histórico de pagamentos disponível até a referência.
- **Bairro e sexo:** recusados; são atributo sensível ou proxy potencialmente discriminatório. Alternativa: auditar desempenho por grupos sem usar esses campos para decidir a fila.
- **Score de bureau:** usado, pois existe antes da referência; sua importância é associação preditiva, não prova de causalidade.
- **Motivos de atraso:** recusados; a base não tem chave de contrato/cliente para ligação e pode conter informação posterior. Alternativa: estruturar o SAC com chave e timestamp auditáveis.

### Model card

- **Dados:** seis CSVs brutos extraídos em 15/03/2026; alvo observado na parcela elegível e features cortadas em `data_referencia`.
- **Uso pretendido:** ordenar contratos candidatos à cobrança preventiva, limitando a fila a 80 por semana.
- **Features:** prazo e valor do contrato, renda tratada, canal, tenure, comprometimento de renda, histórico de compras, pagamentos, SAC e último score de bureau anterior à referência.
- **Limites:** não usar para negar crédito ou definir preço; a prevalência, a política de cobrança e os custos podem mudar. A fila precisa de monitoramento temporal e revisão de fairness.

Antes de enviar, confira o checklist da seção 9 do enunciado.

Faltam ainda dois arquivos que **não** são gerados aqui:
- `decisoes.md` — o que entrou, o que saiu, cada pedido do Ricardo respondido, e o model card.
- `apresentacao.pptx` — 3 slides, para o Ricardo, não para a banca.

> Se algum resultado ficou bom demais, pergunte: *que outra coisa poderia produzir esse mesmo número?*
